# ✅ **NER Lexicon Goals**

For each adjuvant, we need:

### 1. canonical form

### 2. expanded synonyms

### 3. normalization rules

* lowercase
* remove punctuation
* hyphen/space variants
* parentheses-stripped forms
* fused token versions (e.g., "alhydrogel" vs "al hydrogel")

### 4. tokenization-ready forms

(for spaCy, HuggingFace tokenizers, etc.)

### 5. reverse lookup

`mention → adjuvant_vo_id`

This ensures:

* maximum recall
* stable comparisons across tools
* easy matching for LLM outputs

We'll eventually arange
Cell 1 → Build KB  
Cell 2 → Build NER lexicon  
Cell 3 → Load adjuvant_ner_lexicon.csv  
Cell 4 → Run inspection code here

In [1]:
import pandas as pd
import re
from collections import defaultdict
from utils import canonical_normalize, safe_to_list
from utils import fix_mojibake
# -------------------------
# Load your KB
# -------------------------
kb = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/adjuvant_kb.csv", encoding="utf-8-sig")


import ast



# -------------------------
# Normalization helpers
# -------------------------

from utils import canonical_normalize, safe_to_list

def generate_variants(name):
    if not isinstance(name, str) or not name.strip():
        return set()
    base = name.strip().lower()
    variants = {
        base,
        base.replace(" ", ""),      # no-space
        base.replace(" ", "-"),     # hyphenated
        base.replace("-", " "),     # de-hyphenate
        base.replace("/", " "),     # slash -> space
        base.replace("/", ""),      # slash removed
        base.replace("+", " + "),   # spaced plus
        base.replace("+", ""),      # plus removed
        base.replace("_", " "),     # underscore -> space
        base.replace("_", ""),      # underscore removed
    }
    return {canonical_normalize(v) for v in variants if canonical_normalize(v)}

lexicon = []
for _, row in kb.iterrows():
    vo = row["adjuvant_vo_id"]
    pref_raw = row["preferred_name"] if isinstance(row["preferred_name"], str) and row["preferred_name"].strip() else None
    syns_raw = [s for s in safe_to_list(row["synonyms"]) if isinstance(s, str) and s.strip()]

    # store raw names
    all_names = set()
    if pref_raw:
        all_names.add(pref_raw.strip())
    all_names.update(s.strip() for s in syns_raw if s.strip())

    expanded_forms = set()
    for name in all_names:
        expanded_forms.update(generate_variants(name))

    lexicon.append({
        "adjuvant_vo_id": vo,
        "preferred_name": pref_raw,
        "synonyms": sorted(all_names),
        "expanded_forms": sorted(expanded_forms),
        "num_expanded_forms": len(expanded_forms),
    })

lexicon_df = pd.DataFrame(lexicon)
# …save to CSV/JSON as before


lexicon_df.to_csv(
    "Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon.csv",
    index=False, encoding="utf-8-sig"
)

lexicon_df.head()


,adjuvant_vo_id,preferred_name,synonyms,expanded_forms,num_expanded_forms
0,VO_0005271,2B182C,[2B182C],[2b182c],1
1,VO_0005303,2F52,[2F52],[2f52],1
2,VO_0001330,AF03,[AF03],[af03],1
3,VO_0001264,AS-2 vaccine adjuvant,"[AS-2 vaccine adjuvant, AS02, SBAS2, SmithKlin...","[as 2 vaccine adjuvant, as 2vaccineadjuvant, a...",5
4,VO_0001320,AS03,[AS03],[as03],1


In [2]:
import pandas as pd
import re

# Load your KB
kb = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon.csv", encoding="utf-8-sig")

def is_float_like(x):
    try:
        float(str(x))
        return True
    except:
        return False

def has_corrupted_chars(x):
    if not isinstance(x, str):
        return False
    return bool(re.search(r"[�\x96\x97\x91\x92]", x))

def looks_like_description(x):
    if not isinstance(x, str):
        return False
    # If it contains very long sentences or multiple punctuation marks
    if len(x) > 60:
        return True
    if re.search(r"[.;:]{2,}", x):
        return True
    if "adjuvant" in x.lower() and len(x.split()) > 6:
        return True
    return False

# Containers to collect issues
issues = {
    "float_synonyms": [],
    "empty_synonyms": [],
    "corrupted_chars": [],
    "overly_long": [],
    "duplicates_with_spacing": [],
}

# Analyze each row
for idx, row in kb.iterrows():
    syns_raw = row["synonyms"]
    
    # Convert string representation of list into an actual list
    try:
        syns = eval(syns_raw) if isinstance(syns_raw, str) else syns_raw
    except:
        syns = [syns_raw]

    # Track duplicates normalized for comparison
    normalized = {}

    for s in syns:
        if s is None or s == "":
            issues["empty_synonyms"].append((row["adjuvant_vo_id"], s))
            continue

        # Float-like values (e.g., 2.0, 1.5)
        if is_float_like(s) and not isinstance(s, str):
            issues["float_synonyms"].append((row["adjuvant_vo_id"], s))

        # Corrupted characters
        if has_corrupted_chars(s):
            issues["corrupted_chars"].append((row["adjuvant_vo_id"], s))

        # Description-like values slipping into synonyms
        if looks_like_description(s):
            issues["overly_long"].append((row["adjuvant_vo_id"], s))

        # Check duplicates based on stripped/lowercase
        norm = s.strip().lower() if isinstance(s, str) else str(s).lower()
        if norm in normalized:
            issues["duplicates_with_spacing"].append((row["adjuvant_vo_id"], normalized[norm], s))
        else:
            normalized[norm] = s

# Pretty print results
print("\n=== FLOAT SYNONYMS ===")
for item in issues["float_synonyms"][:20]:
    print(item)
print(f"Total: {len(issues['float_synonyms'])}")

print("\n=== EMPTY SYNONYMS ===")
for item in issues["empty_synonyms"][:20]:
    print(item)
print(f"Total: {len(issues['empty_synonyms'])}")

print("\n=== CORRUPTED CHARACTER SYNONYMS ===")
for item in issues["corrupted_chars"][:20]:
    print(item)
print(f"Total: {len(issues['corrupted_chars'])}")

print("\n=== OVERLY LONG / DESCRIPTION-LIKE SYNONYMS ===")
for item in issues["overly_long"][:20]:
    print(item)
print(f"Total: {len(issues['overly_long'])}")

print("\n=== DUPLICATE SYNONYMS (DIFFERENT SPACING/CASE) ===")
for item in issues["duplicates_with_spacing"][:20]:
    print(item)
print(f"Total: {len(issues['duplicates_with_spacing'])}")



=== FLOAT SYNONYMS ===
Total: 0

=== EMPTY SYNONYMS ===
Total: 0

=== CORRUPTED CHARACTER SYNONYMS ===
Total: 0

=== OVERLY LONG / DESCRIPTION-LIKE SYNONYMS ===
('VO_0001288', 'N, N-dioctadecyl-Nâ€², Nâ€²-bis( 2-hydroxyethyl) propanediamine')
('VO_0001261', 'Dimethyldioctadecylammonium bromide, Dimethy1dioctadecylammonium bromide')
('VO_0001261', 'dimethy1distearylammonium bromide (CAS Registry Number 3700-67-2).')
('VO_0001341', 'DL-PGL (Polyester poly (DL-lactide-co-glycolide)) vaccine adjuvant')
('VO_0001341', 'Polyester poly (DL-lactide-co-glycolide), PLGA, PLGA microspheres')
('VO_0001295', 'N-acetylglucosaminyl-(b1-4)-N-acetylmuramyl-L-alanyl-D-isoglutamine (CAS Registry Number 70280-03-4)')
('VO_0001343', 'Immunoliposomes Containing Antibodies to Costimulatory Molecules')
('VO_0001343', 'Immunoliposomes prepared from Dehydration-Rehyrdation Vesicles (DRVs)')
Total: 8

=== DUPLICATE SYNONYMS (DIFFERENT SPACING/CASE) ===
('VO_0000127', 'Aluminum Hydroxide', 'Aluminum hydroxide')


#### FULL TOKEN + SUFFIX DETECTION 
1. Build token frequencies for all synonyms + preferred names
2. Build suffix (1–3 word) frequencies
3. Identify boilerplate candidates based on:
4. High frequency across VO IDs
5. High cross-adjuvant distribution
6. Output a ranked list of boilerplate tokens and suffixes
7. Save results to CSV for review

In [3]:
import pandas as pd
import re
from collections import Counter, defaultdict

# ------------------------------------------
# LOAD LEXICON (adjuvant_kb.csv)
# ------------------------------------------
kb = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon.csv", encoding="utf-8-sig")

# Convert synonyms column into Python lists if needed
def to_list(x):
    if isinstance(x, str):
        if x.startswith("[") and x.endswith("]"):
            try:
                return eval(x)
            except:
                return [x]
        return [x]
    if isinstance(x, list):
        return x
    return []

kb["synonyms"] = kb["synonyms"].apply(to_list)
kb["preferred_name"] = kb["preferred_name"].fillna("")

# ------------------------------------------
# BUILD MASTER NAME LIST
# ------------------------------------------
all_names = []
vo_id_map = defaultdict(set)  # token -> list of VO IDs where it appears

for _, row in kb.iterrows():
    vo = row["adjuvant_vo_id"]
    names = [row["preferred_name"]] + row["synonyms"]

    for name in names:
        if not isinstance(name, str): 
            continue
        clean = name.strip().lower()
        if clean:
            all_names.append((vo, clean))
            vo_id_map[clean].add(vo)

# ------------------------------------------
# TOKENIZE NAMES INTO WORDS
# ------------------------------------------
token_counter = Counter()
token_vo_distribution = defaultdict(set)

for vo, name in all_names:
    tokens = re.findall(r"[a-zA-Z0-9\-\+]+", name.lower())
    for t in tokens:
        token_counter[t] += 1
        token_vo_distribution[t].add(vo)

# ------------------------------------------
# Identify High-Frequency Boilerplate Tokens
# ------------------------------------------
total_adjuvants = kb["adjuvant_vo_id"].nunique()

boilerplate_tokens = []
for token, count in token_counter.items():
    spread = len(token_vo_distribution[token])
    freq_ratio = count / len(all_names)
    spread_ratio = spread / total_adjuvants

    # Strong heuristics for boilerplate detection
    if (
        spread_ratio > 0.25      # appears in >25% of all adjuvants
        and freq_ratio > 0.02    # appears in >2% of all occurrences
        and len(token) > 2       # ignore chemical symbols (e.g., "cg", "ol")
        and not re.search(r"\d", token)  # avoid eliminating numeric chemicals
    ):
        boilerplate_tokens.append((token, count, spread))

# Sort by importance
boilerplate_tokens = sorted(boilerplate_tokens, key=lambda x: (-x[2], -x[1]))

# ------------------------------------------
# SUFFIX ANALYSIS (1–3 word n-grams)
# ------------------------------------------
suffix_counter = Counter()
suffix_vo_distribution = defaultdict(set)

def extract_suffixes(name):
    tokens = name.split()
    suffixes = []
    # last 1 word
    if len(tokens) >= 1:
        suffixes.append(tokens[-1])
    # last 2 words
    if len(tokens) >= 2:
        suffixes.append(" ".join(tokens[-2:]))
    # last 3 words
    if len(tokens) >= 3:
        suffixes.append(" ".join(tokens[-3:]))
    return suffixes

for vo, name in all_names:
    for s in extract_suffixes(name.lower()):
        suffix_counter[s] += 1
        suffix_vo_distribution[s].add(vo)

suffix_boilerplates = []
for suf, count in suffix_counter.items():
    spread = len(suffix_vo_distribution[suf])
    spread_ratio = spread / total_adjuvants

    if (
        spread_ratio > 0.20        # appears widely across adjuvants
        and len(suf.split()) <= 3  # only 1–3 word suffixes
        and len(suf) < 35          # exclude long sentences
    ):
        suffix_boilerplates.append((suf, count, spread))

suffix_boilerplates = sorted(suffix_boilerplates, key=lambda x: (-x[2], -x[1]))

# ------------------------------------------
# SAVE OUTPUT
# ------------------------------------------

bp_tokens_df = pd.DataFrame(
    boilerplate_tokens, 
    columns=["token", "total_occurrences", "distinct_vo_ids"]
)
bp_tokens_df.to_csv(
    "Dataset/VIOLIN_12-10-2025/interim/boilerplate_tokens.csv",
    index=False, encoding="utf-8-sig"
)

bp_suffix_df = pd.DataFrame(
    suffix_boilerplates, 
    columns=["suffix", "total_occurrences", "distinct_vo_ids"]
)
bp_suffix_df.to_csv(
    "Dataset/VIOLIN_12-10-2025/interim/boilerplate_suffixes.csv",
    index=False, encoding="utf-8-sig"
)

print("=== BOILERPLATE TOKEN DETECTION COMPLETE ===")
print(bp_tokens_df.head(20))

print("\n=== BOILERPLATE SUFFIX DETECTION COMPLETE ===")
print(bp_suffix_df.head(20))


=== BOILERPLATE TOKEN DETECTION COMPLETE ===
      token  total_occurrences  distinct_vo_ids
0  adjuvant                 94               44
1   vaccine                 61               29

=== BOILERPLATE SUFFIX DETECTION COMPLETE ===
             suffix  total_occurrences  distinct_vo_ids
0          adjuvant                 92               43
1  vaccine adjuvant                 59               28


In [4]:
import re

GENERIC_EXACT = {
    "adjuvant",
    "vaccine",
    "vaccine adjuvant",
    "adjuvant component",
    "component",
    "vaxjo adjuvant",
    "vaccine component",
}

# Heuristic: how long can we tolerate a "name" for NER?
# You can tighten/loosen this later.
MAX_NAME_CHARS = 80

PUBMED_PATTERN = re.compile(
    r"\[?\s*pubmed[:\s]*\d{3,10}[^\]]*\]?", 
    flags=re.I
)

def strip_pubmed_bits(text: str) -> str:
    """
    Remove [PubMed: 123456] or similar citation fragments.
    Leave other content intact.
    """
    # Remove pure bracketed or inline PubMed references
    cleaned = PUBMED_PATTERN.sub("", text)
    # Collapse excessive whitespace, strip punctuation at ends
    cleaned = re.sub(r"\s+", " ", cleaned).strip(" .;:,[]()")
    return cleaned.strip()

def looks_sentence_like(s: str) -> bool:
    """
    Rough heuristic: if there is a period mid-string, or many spaces,
    or it's very long, treat as a description rather than a label.
    """
    if len(s) > MAX_NAME_CHARS:
        return True
    # Too many words is also a red flag
    if len(s.split()) > 10:
        return True
    # Internal full stop suggests sentence
    if "." in s[:-1]:
        return True
    return False

def clean_string_list(items):
    cleaned = []
    seen_lower = set()

    for raw in items:
        if not isinstance(raw, str):
            continue

        s = raw.strip()
        s = fix_mojibake(s)

        if not s:
            continue

        # 1) Strip PubMed bits while trying to preserve the core label
        s_no_pm = strip_pubmed_bits(s)

        # If everything disappears, skip
        if not s_no_pm:
            continue

        # If even after stripping we still see "pubmed" => junk, drop
        if "pubmed" in s_no_pm.lower():
            continue

        # 2) Drop generic boilerplate
        if s_no_pm.lower() in GENERIC_EXACT:
            continue

        # 3) Sentence-like and still long => likely description; drop for NER
        if looks_sentence_like(s_no_pm):
            continue

        # 4) Deduplicate (case-insensitive) on the final cleaned string
        key = s_no_pm.lower()
        if key in seen_lower:
            continue
        seen_lower.add(key)
        cleaned.append(s_no_pm)

    return cleaned

lexicon_df["synonyms"] = lexicon_df["synonyms"].apply(clean_string_list)
lexicon_df["expanded_forms"] = lexicon_df["expanded_forms"].apply(clean_string_list)

lexicon_df = lexicon_df[
    (lexicon_df["preferred_name"].notna()) |
    (lexicon_df["synonyms"].apply(len) > 0)
].reset_index(drop=True)

'''lexicon_df.to_json(
    "Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon_clean.json",
    orient="records",
    indent=2, ensure_ascii=False
)'''
import json
from pathlib import Path


Path("Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon_clean.json").write_text(
    json.dumps(lexicon_df.to_dict(orient="records"), indent=2, ensure_ascii=False),
    encoding="utf-8"
)

lexicon_df.to_csv(
    "Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon_clean.csv",
    index=False, encoding="utf-8-sig"
)


In [5]:
#Validate the final lexicon and generate NER coverage stats
import pandas as pd
import numpy as np

# Load cleaned lexicon
lex = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/adjuvant_ner_lexicon_clean.csv", encoding="utf-8-sig")

# Parse list-like columns
import ast

def to_list(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        s = x.strip()
        if s.startswith("[") and s.endswith("]"):
            try:
                val = ast.literal_eval(s)
                return val if isinstance(val, list) else [val]
            except Exception:
                return []
        return [s] if s else []
    return []


lex["synonyms"] = lex["synonyms"].apply(to_list)
lex["expanded_forms"] = lex["expanded_forms"].apply(to_list)

# ---------------------------
# Summary statistics
# ---------------------------
total_adj = lex["adjuvant_vo_id"].nunique()
avg_synonyms = lex["synonyms"].apply(len).mean()
avg_expanded = lex["expanded_forms"].apply(len).mean()
min_syn = lex["synonyms"].apply(len).min()
max_syn = lex["synonyms"].apply(len).max()

print("=== LEXICON SUMMARY ===")
print(f"Total adjuvant entries: {total_adj}")
print(f"Average synonyms per adjuvant: {avg_synonyms:.2f}")
print(f"Average expanded forms per adjuvant: {avg_expanded:.2f}")
print(f"Min synonyms: {min_syn}")
print(f"Max synonyms: {max_syn}")

# ---------------------------
# Identify weak coverage (0–1 synonyms)
# ---------------------------
weak = lex[lex["synonyms"].apply(len) <= 1]
print("\n=== ADJUVANTS WITH VERY FEW SYNONYMS (<=1) ===")
print(weak[["adjuvant_vo_id", "preferred_name", "synonyms"]].head(20))

# ---------------------------
# Top 20 richest synonym sets
# ---------------------------
lex["syn_count"] = lex["synonyms"].apply(len)
print("\n=== TOP 20 SYNONYM-RICH ADJUVANTS ===")
print(lex.sort_values("syn_count", ascending=False)[
    ["adjuvant_vo_id", "preferred_name", "syn_count"]
].head(20))

# ---------------------------
# Expanded form distribution
# ---------------------------
lex["exp_count"] = lex["expanded_forms"].apply(len)
print("\n=== EXPANDED FORM DISTRIBUTION ===")
print(lex["exp_count"].describe())

# ---------------------------
# Sanity Check: critical adjuvants
# ---------------------------
critical = [
    "alhydrogel",
    "aluminum hydroxide",
    "aluminum phosphate",
    "mpl",
    "cpg",
    "freund",
    "montanide",
    "cholera toxin",
    "lt(r192g)"
]

print("\n=== CRITICAL ADJUVANTS CHECK ===")
for c in critical:
    hits = lex[
        lex["preferred_name"].str.lower().str.contains(c, na=False) |
        lex["synonyms"].apply(lambda lst: any(c in s.lower() for s in lst))
    ]
    print(f"\n--- {c} ---")
    if hits.empty:
        print("MISSING — problem!")
    else:
        print(hits[["adjuvant_vo_id", "preferred_name", "synonyms"]].head())


=== LEXICON SUMMARY ===
Total adjuvant entries: 101
Average synonyms per adjuvant: 1.80
Average expanded forms per adjuvant: 2.95
Min synonyms: 1
Max synonyms: 7

=== ADJUVANTS WITH VERY FEW SYNONYMS (<=1) ===
   adjuvant_vo_id                                   preferred_name  \
0      VO_0005271                                           2B182C   
1      VO_0005303                                             2F52   
2      VO_0001330                                             AF03   
4      VO_0001320                                             AS03   
5      VO_0001265                                             AS04   
6      VO_0001334                      Abisco-100 vaccine adjuvant   
9      VO_0005302                                     Adjuplex+GLA   
10     VO_0005207                                            Advax   
11     VO_0001335  Albumin-heparin microparticles vaccine adjuvant   
16     VO_0000191                                        Arlacel A   
19     VO_0001287   

C:\Users\hasin.rehana\AppData\Local\Temp\ipykernel_14668\2922068010.py:86: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  lex["preferred_name"].str.lower().str.contains(c, na=False) |


In [6]:
import pandas as pd
import re
from utils import fix_mojibake

kb = pd.read_csv("Dataset/VIOLIN_12-10-2025/interim/adjuvant_kb.csv", encoding="utf-8-sig")

pmids = set()

# 1. From structured pmid column
if "pmids" in kb.columns:
    for lst in kb["pmids"].dropna():
        try:
            for p in eval(lst):
                pmids.add(str(p))
        except:
            pass

# 2. Extract PMIDs from any text fields as fallback
PMID_RE = re.compile(r"\b(\d{5,10})\b")

text_cols = [c for c in kb.columns if kb[c].dtype == object]

for col in text_cols:
    for val in kb[col].dropna():
        for p in PMID_RE.findall(str(val)):
            pmids.add(p)

print("Total unique PMIDs found:", len(pmids))


Total unique PMIDs found: 1343
